In [1]:
import json
import time
import re
import urllib
from datetime import date
import polars as pl
import numpy as np
import textdistance as td
import requests
from bs4 import BeautifulSoup
import nltk
from nltk.stem.cistem import Cistem
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/tobias/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
schema={
    'id': pl.Utf8,
    'abstract': pl.Utf8,
    'beratungsstand': pl.Utf8,
    'vorgangstyp': pl.Utf8,
    'gesta': pl.Utf8,
    'archiv': pl.Utf8,
    'sachgebiet': pl.List(pl.Utf8),
    'typ': pl.Utf8,
    'wahlperiode': pl.UInt64,
    'zustimmungsbeduerftigkeit': pl.List(pl.Utf8),
    'initiative': pl.List(pl.Utf8),
    'deskriptor': pl.List(pl.Struct([
        pl.Field('fundstelle', pl.Boolean),
        pl.Field('name', pl.Utf8),
        pl.Field('typ', pl.Utf8),
    ])),
    'aktualisiert': pl.Utf8,
    'titel': pl.Utf8,
    'datum': pl.Utf8,
    'verkuendung': pl.List(pl.Struct([
        pl.Field('verkuendungsdatum', pl.Utf8)
    ]))
}
df_vorgang = pl.DataFrame(json.load(open('vorgang.json', encoding='utf-8')), schema=schema)
df_gesetze = df_vorgang.filter(pl.col('vorgangstyp') == 'Gesetzgebung')
pl.Config(fmt_str_lengths=100)
df_gesetze['beratungsstand'].unique()

beratungsstand
str
"""Einbringung abgelehnt"""
"""Den Ausschüssen zugewiesen"""
"""Verkündet"""
"""Abgelehnt"""
"""Dem Bundestag zugeleitet - Noch nicht beraten"""
…
"""Dem Bundesrat zugeleitet - Noch nicht beraten"""
"""Bundesrat hat Zustimmung versagt"""
"""Noch nicht beraten"""


In [3]:
df_vorgangspos = pl.read_json('vorgangsposition.json', infer_schema_length=40000)
df_vorgangspos = df_vorgangspos.with_columns(
   pl.col("datum").str.to_date(format="%Y-%m-%d")
)
df_vorgangspos_gesetzgebung = df_vorgangspos.filter((pl.col('vorgangstyp') == 'Gesetzgebung') & (pl.col('typ') == 'Vorgangsposition'))
df_vorgang_duration = df_vorgangspos_gesetzgebung.group_by(['vorgang_id']).agg([
    pl.col('datum').min().alias('vorgang_start'), 
])

In [4]:
df_gesetze.head()

id,abstract,beratungsstand,vorgangstyp,gesta,archiv,sachgebiet,typ,wahlperiode,zustimmungsbeduerftigkeit,initiative,deskriptor,aktualisiert,titel,datum,verkuendung
str,str,str,str,str,str,list[str],str,u64,list[str],list[str],list[struct[3]],str,str,str,list[struct[1]]
"""338789""","""Dämpfung der Kostenbelastungen für stromverbrauchende Unternehmen und Haushalte durch die ansteigend…","""Beschlussempfehlung liegt vor""","""Gesetzgebung""","""E048""",null,"[""Energie"", ""Wirtschaft""]","""Vorgang""",21,"[""Nein, laut Gesetzentwurf (Drs 509/26 (bes.eilbed.))""]","[""Bundesregierung""]","[{false,""Bundeszuschuss"",""Sachbegriffe""}, {false,""Elektrizitätswirtschaft"",""Sachbegriffe""}, … {false,""Unternehmen"",""Sachbegriffe""}]","""2026-11-13T13:36:04+01:00""","""Gesetz zur Änderung des Energiewirtschaftsgesetzes zur Gewährung eines Zuschusses zu den Übertragung…","""2026-11-13""",null
"""338785""","""<strong>Beschlussempfehlung des Ausschusses:</strong> Änderungen""","""Verabschiedet""","""Gesetzgebung""","""I012""",null,"[""Bildung und Erziehung""]","""Vorgang""",21,"[""Ja, laut Gesetzentwurf (Drs 505/26 (bes.eilbed.))""]","[""Bundesregierung""]","[{false,""Finanzausgleichsgesetz"",""Rechtsmaterialien""}, {false,""Ganztagsschule"",""Sachbegriffe""}, … {true,""Startchancen- und Qualitätsentwicklungsgesetz"",""Rechtsmaterialien""}]","""2026-11-13T14:13:44+01:00""","""Gesetz zur Verbesserung der Startchancen von Kindern und zur Qualitätsentwicklung in der Kindertages…","""2026-11-13""",null
"""338284""","""Einwilligung oder nicht ausdrücklich erfolgter Widerspruch des möglichen Spenders als Voraussetzung …","""Überwiesen""","""Gesetzgebung""","""M016""",null,"[""Gesundheit"", ""Recht""]","""Vorgang""",21,"[""Nein, laut Gesetzentwurf (Drs 21/7570)""]",null,"[{false,""Familie"",""Sachbegriffe""}, {false,""Gesundheitserziehung"",""Sachbegriffe""}, … {false,""Öffentlichkeitsarbeit der Bundesregierung"",""Sachbegriffe""}]","""2026-11-13T15:31:38+01:00""","""Viertes Gesetz zur Änderung des Transplantationsgesetzes - Einführung einer Widerspruchsregelung""","""2026-11-13""",null
"""335476""",null,"""Verabschiedet""","""Gesetzgebung""","""B048""",null,"[""Innere Sicherheit"", ""Medien, Kommunikation und Informationstechnik""]","""Vorgang""",21,"[""Ja, laut Gesetzentwurf (Drs 323/26 (bes.eilbed.))""]","[""Bundesregierung""]","[{false,""BSI-Gesetz"",""Rechtsmaterialien""}, {false,""Brief-, Post- und Fernmeldegeheimnis"",""Sachbegriffe""}, … {false,""Telekommunikation-Digitale-Dienste-Datenschutz-Gesetz"",""Rechtsmaterialien""}]","""2026-11-13T15:14:30+01:00""","""Gesetz zur Stärkung der Cybersicherheit""","""2026-11-13""",null
"""335444""","""Maßnahmen zur Modernisierung und Entbürokratisierung der Fahrschulausbildung, zur Reduzierung der Ko…","""Verabschiedet""","""Gesetzgebung""","""J021""",null,"[""Verkehr""]","""Vorgang""",21,"[""Ja, laut Gesetzentwurf (Drs 330/26)""]","[""Bundesregierung""]","[{false,""Behörde"",""Sachbegriffe""}, {false,""Berufliche Qualifikation"",""Sachbegriffe""}, … {false,""Verwaltungsverfahren"",""Sachbegriffe""}]","""2026-11-13T13:40:50+01:00""","""Gesetz zur Änderung des Fahrlehrergesetzes und anderer straßenverkehrsrechtlicher Gesetze""","""2026-11-13""",null


In [5]:
df_gesetze = df_gesetze.join(df_vorgang_duration.select(['vorgang_id', 'vorgang_start']), left_on='id', right_on='vorgang_id', how='left')

In [6]:
#df_vorgang_docs = df_vorgangspos.filter(pl.col('dokumentart') == 'Drucksache').group_by('vorgang_id').agg(pl.col('fundstelle').struct.field("dokumentnummer").explode())
#df_gesetze = df_gesetze.join(df_vorgang_docs.select(['vorgang_id', 'dokumentnummer']), left_on='id', right_on='vorgang_id', how='left')

In [7]:
today = date.today()
df_gesetze = df_gesetze.with_columns(
    pl.when(pl.col('beratungsstand') == 'Verkündet').then(
        pl.col('verkuendung').list.eval(pl.col('').struct.field('verkuendungsdatum').str.to_date(format="%Y-%m-%d")).list.min()
    ).when(pl.col('beratungsstand') == 'Abgelehnt').then(
        pl.col('datum').str.to_date(format="%Y-%m-%d")
    ).otherwise(today).alias('vorgang_end')
)
df_gesetze = df_gesetze.with_columns((pl.col('vorgang_end') - pl.col('vorgang_start')).dt.total_days().alias('vorgangsdauer'))

In [8]:
reg_frag = ['Fraktion der CDU/CSU', 'Fraktion der SPD']

def fix_initiative(x):
    return pl.when(x.list.set_intersection(reg_frag).len == len(reg_frag)).then(
        x.list.set_difference(reg_frag)
    ).otherwise(x)

df_gesetze = df_gesetze.with_columns(pl.when(pl.col('initiative').list.set_intersection(reg_frag).list.len() == len(reg_frag)).then(
        pl.col('initiative').list.set_difference(reg_frag).list.concat(pl.Series(['Bundesregierung']))
    ).otherwise(pl.col('initiative')).alias('initiative'))

In [9]:
df_gesetze = df_gesetze.with_columns(pl.col('deskriptor').list.eval(pl.element().struct.field('name')).alias('keywords'))

In [10]:
df_gesetze.select(pl.col("*").exclude("abstract", "vorgangstyp", "gesta", "archiv", "typ", "deskriptor")).write_json('gesetze.json')

In [11]:
df_gesetze_public = df_gesetze.filter(pl.col('beratungsstand') == 'Verkündet')
bins = [7, 14, 30, 60, 120, 240, 365]
duration_bin = np.digitize(df_gesetze_public.select('vorgangsdauer').to_series(), bins, right=True)
df_gesetze_public.select('vorgangsdauer').to_series().hist(bins)

breakpoint,category,count
f64,cat,u32
14.0,"""[7.0, 14.0]""",1
30.0,"""(14.0, 30.0]""",1
60.0,"""(30.0, 60.0]""",15
120.0,"""(60.0, 120.0]""",61
240.0,"""(120.0, 240.0]""",89
365.0,"""(240.0, 365.0]""",10


In [12]:
df_gesetze_public_bins = df_gesetze_public.with_columns(pl.from_numpy(duration_bin, schema=['duration_bin']))
df_gesetze_public_bins.filter(pl.col('duration_bin') == pl.col('duration_bin').min())

id,abstract,beratungsstand,vorgangstyp,gesta,archiv,sachgebiet,typ,wahlperiode,zustimmungsbeduerftigkeit,initiative,deskriptor,aktualisiert,titel,datum,verkuendung,vorgang_start,vorgang_end,vorgangsdauer,keywords,duration_bin
str,str,str,str,str,str,list[str],str,u64,list[str],list[str],list[struct[3]],str,str,str,list[struct[1]],date,date,i64,list[str],i64
"""332608""","""Maßnahmen zur Begegnung der Preisvolatilität an öffentlichen Tankstellen und zur Erhöhung der Preisv…","""Verkündet""","""Gesetzgebung""","""E026""",null,"[""Wirtschaft""]","""Vorgang""",21,"[""Nein, laut Gesetzentwurf (Drs 21/4744)"", ""Nein, laut Verkündung (BGBl I)""]","[""Bundesregierung""]","[{false,""Benzin"",""Sachbegriffe""}, {false,""Beweislast"",""Sachbegriffe""}, … {false,""Wettbewerb"",""Sachbegriffe""}]","""2026-04-16T15:06:36+02:00""","""Gesetz zur Einführung eines Gesetzes zur Anpassung von Kraftstoffpreisen und zur Änderung des Gesetz…","""2026-03-27""","[{""2026-03-31""}]",2026-03-17,2026-03-31,14,"[""Benzin"", ""Beweislast"", … ""Wettbewerb""]",1


In [13]:
df_gesetze_public_bins.filter(pl.col('duration_bin') == pl.col('duration_bin').max())

id,abstract,beratungsstand,vorgangstyp,gesta,archiv,sachgebiet,typ,wahlperiode,zustimmungsbeduerftigkeit,initiative,deskriptor,aktualisiert,titel,datum,verkuendung,vorgang_start,vorgang_end,vorgangsdauer,keywords,duration_bin
str,str,str,str,str,str,list[str],str,u64,list[str],list[str],list[struct[3]],str,str,str,list[struct[1]],date,date,i64,list[str],i64
"""326806""","""<strong>Beschlussempfehlung des Ausschusses:</strong> Änderungen; Annahme einer Entschließung""","""Verkündet""","""Gesetzgebung""","""B021""",null,"[""Innere Sicherheit""]","""Vorgang""",21,"[""Ja, laut Gesetzentwurf (Drs 557/25)"", ""Ja, laut Verkündung (BGBl I)""]","[""Bundesregierung""]","[{false,""AZR-Gesetz"",""Rechtsmaterialien""}, {false,""AZRG-Durchführungsverordnung"",""Rechtsmaterialien""}, … {false,""Zollverwaltungsgesetz"",""Rechtsmaterialien""}]","""2026-10-28T10:31:45+01:00""","""Gesetz zur Modernisierung des Bundespolizeigesetzes""","""2026-09-25""","[{""2026-10-27""}]",2025-10-10,2026-10-27,382,"[""AZR-Gesetz"", ""AZRG-Durchführungsverordnung"", … ""Zollverwaltungsgesetz""]",7
"""325563""","""Umsetzung europäischer Vorgaben zur Sicherstellung des Zugangs für Schuldner zu unabhängigen Schuldn…","""Verkündet""","""Gesetzgebung""","""C031""",null,"[""Gesellschaftspolitik, soziale Gruppen"", ""Recht""]","""Vorgang""",21,"[""Ja, laut Gesetzentwurf (Drs 436/25 (bes.eilbed.))"", ""Ja, laut Verkündung (BGBl I)""]","[""Bundesregierung""]","[{false,""Beratungsstelle"",""Sachbegriffe""}, {false,""Bericht"",""Sachbegriffe""}, … {true,""Schuldnerberatungsdienstegesetz"",""Rechtsmaterialien""}]","""2026-10-23T13:05:58+02:00""","""Gesetz über den Zugang zu Schuldnerberatungsdiensten für Verbraucher (Schuldnerberatungsdienstegeset…","""2026-09-25""","[{""2026-10-15""}]",2025-09-05,2026-10-15,405,"[""Beratungsstelle"", ""Bericht"", … ""Schuldnerberatungsdienstegesetz""]",7
"""321949""","""Stärkung von Reallaboren als wichtigem Instrument der Innovationsförderung und des regulatorischen L…","""Verkündet""","""Gesetzgebung""","""E003""",null,"[""Wirtschaft"", ""Wissenschaft, Forschung und Technologie""]","""Vorgang""",21,"[""Nein, laut Gesetzentwurf (Drs 21/218)"", ""Ja, laut Bundestag (Drs 21/6698)"", ""Ja, laut Verkündung (BGBl I)""]","[""Bundesregierung""]","[{false,""Arztpraxis"",""Sachbegriffe""}, {false,""Bericht der Bundesregierung"",""Sachbegriffe""}, … {false,""Wissenstransfer"",""Sachbegriffe""}]","""2026-10-09T15:46:49+02:00""","""Gesetz zur Stärkung des Innovations- und Wirtschaftsstandorts Deutschland durch Erprobungsfreiräume""","""2026-09-25""","[{""2026-10-09""}]",2025-05-20,2026-10-09,507,"[""Arztpraxis"", ""Bericht der Bundesregierung"", … ""Wissenstransfer""]",7


In [14]:
def map_gesetze(tokens):
    overlap = df_gesetze.with_columns(pl.col('titel.stem').map_elements(lambda x: td.lcsstr.normalized_similarity(tokens, x), return_dtype=pl.Float64).alias('similarity')).select(['titel', 'titel.stem', 'id', 'similarity'])
    overlap_ = overlap.sort('similarity', descending=True)
    overlap = overlap_.filter(pl.col('similarity') > 0.5)
    if overlap.shape[0] == 0:
        print(f'Warning: Count not find any Gesetze with similarity > 0.5: {tokens}')
        print(overlap_.head())
        return None
    if overlap.shape[0] > 1:
        if overlap.row(1, named=True)['similarity'] * 2 > overlap.row(0, named=True)['similarity'] and overlap.row(0, named=True)['similarity'] != 1:
            print(f'Error: could not map gesetz: {tokens}')
            print(overlap.head())
            return None
    #print(tokens)
    #print(overlap.head())
    return overlap.row(0, named=True)

bad_words = [
    'beratung',
    'gebrach',
    'entwurf',
    'eine'
]
stopwords_german = stopwords.words('german')
def rm_stop_words(txt):
    words = txt.split()
    words = list(filter(lambda x: x not in stopwords_german, words))
    return ' '.join(words)
    
def pre_process(txt):
    txt = txt.split('eingebrach', 1)[1]
    txt = ' '.join(txt.split())
    txt = re.sub(r'[a-z]\)\(', '', txt)
    txt = re.sub(r'des von (der|den) .*(fraktio|bundesregierung)', '', txt)
    txt = re.sub(r'^–', '', txt)
    for word in bad_words:
        txt = txt.replace(word, '', 1)
    return rm_stop_words(txt)

stemmer = Cistem()
def stem(txt):
    words = []
    for word in txt.split():
        words.append(stemmer.segment(word)[0])
    return ' '.join(words)

def parse_top_details(details):
    top_l = []
    for top in str(details).split('<br/><br/>'):
        top = BeautifulSoup(top, 'lxml')
        vorgang_id = None
        for a in top.find_all('a'):
            drucksache = df_vorgangspos.filter((pl.col('dokumentart') == 'Drucksache') & (pl.col('fundstelle').struct.field("drucksachetyp") == 'Gesetzentwurf') & (pl.col('fundstelle').struct.field("pdf_url") == a['href']))
            if drucksache.shape[0] == 0:
                continue
            if drucksache.shape[0] == 1:
                vorgang_id_new = drucksache.row(0, named=True)['vorgang_id']
                if vorgang_id and vorgang_id != vorgang_id_new:
                    gesetz = df_gesetze.filter(pl.col('id') == vorgang_id).row(0, named=True)
                    print('* ' + gesetz['titel'])
                    top_l.append(gesetz)
                    # warning: more then one gesetz discussed in one TOP
                vorgang_id = vorgang_id_new
            if drucksache.shape[0] > 1:
                print(drucksache)
                raise RuntimeError('Found multiple vorgang ids for document')
        if vorgang_id:
            gesetz = df_gesetze.filter(pl.col('id') == vorgang_id).row(0, named=True)
            print('* ' + gesetz['titel'])
            top_l.append(gesetz)
        else:
            txt_ = top.get_text(' ')
            for txt in txt_.split('Drucksache'):
                if 'gesetz' in txt.lower() and 'entwurf' in txt.lower():
                    txt = stem(txt)
                    txt = pre_process(txt)
                    gesetz = map_gesetze(txt)
                    if gesetz is not None:
                        print('* ' + gesetz['titel'])
                        top_l.append(gesetz)
    return top_l

def map_top_gesetze(soup):
    tagesordnung = {}
    for conf in soup.find_all('div', {'class': 'bt-conference-title'}):
        sitzung = conf.text.strip()
        print(sitzung)
        top_l = []
        for thema in conf.parent.parent.find_all('td', {'data-th': 'Thema'}):
            if thema.p:
                top_l += parse_top_details(thema.p)
        print()
        tagesordnung[sitzung] = top_l
    return tagesordnung

def get_tagesordnung_gesetze_html(kw):
    res = requests.get('https://www.bundestag.de/apps/plenar/plenar/conferenceweekDetail.form', {'year': 2025, 'week': kw, 'limit': 10})
    time.sleep(0.3)
    soup = BeautifulSoup(res.text, 'lxml')
    return map_top_gesetze(soup)

def get_tagesordnung_gesetze_json(kw=None, year=None):
    args = None
    if kw and year:
        args = {'year': year, 'week': kw}
    res = requests.get('https://www.bundestag.de/apps/plenar/plenar/conferenceWeekJSON', args)
    time.sleep(0.3)
    return res.json()

def get_tagesordnung_gesetze():
    tagesordnung = {}
    current_week = get_tagesordnung_gesetze_json()
    conferences = current_week["conferences"]
    if "previous" in current_week:
        prev_week = get_tagesordnung_gesetze_json(current_week["previous"]["week"], current_week["previous"]["year"])
        conferences += prev_week["conferences"]
    if "next" in current_week:
        next_week = get_tagesordnung_gesetze_json(current_week["next"]["week"], current_week["next"]["year"])
        conferences += next_week["conferences"]
    conferences = sorted(conferences, key=lambda x: x["conferenceNumber"])
    for day in conferences:
        sitzung = f'{day["conferenceDate"]["date"]} ({day["conferenceNumber"]}. Sitzung)'
        print(sitzung)
        top_l = []
        for top in day["rows"]:
            topic = top["topic"]
            title = topic["title"]
            detail = None
            link = None
            if "detail" in topic:
                detail = topic["detail"]
            if "link" in topic:
                link = topic["link"]
            top_l += parse_top_details(detail)
        tagesordnung[sitzung] = top_l
        print()
    return tagesordnung
    

In [15]:
df_gesetze = df_gesetze.with_columns(pl.col('titel').map_elements(stem, return_dtype=pl.self_dtype()).map_elements(rm_stop_words, return_dtype=pl.self_dtype()).alias('titel.stem'))

In [16]:
#df_vorgangspos.filter((pl.col('dokumentart') == 'Drucksache')).select(pl.col('fundstelle').struct.field("pdf_url"))
#df_vorgangspos.filter(pl.col('fundstelle').struct.field("pdf_url").eq('https://dserver.bundestag.de/btd/20/106/2010664.pdf'))

In [17]:
top = get_tagesordnung_gesetze()

4. November 2026 (104. Sitzung)

5. November 2026 (105. Sitzung)
* Gesetz zur Neuordnung der Pflegeversicherung (Pflegeneuordnungsgesetz)
* Gesetz zur besseren Bekämpfung der Kriminalität von Kindern, Jugendlichen und Heranwachsenden
* Gesetz zu dem Protokoll vom 9. Juni 2026 zur Änderung des Abkommens vom 6. April 2010 zwischen der Bundesrepublik Deutschland und der Republik Albanien zur Vermeidung der Doppelbesteuerung und der Steuerverkürzung auf dem Gebiet der Steuern vom Einkommen und vom Vermögen
* Gesetz zu dem Vertrag vom 29. April 2024 zwischen der Bundesrepublik Deutschland und dem Königreich Belgien über die Polizei- und Zollzusammenarbeit (Deutsch-Belgischer Polizeivertrag)
* Gesetz zur Änderung des Fahrlehrergesetzes und anderer straßenverkehrsrechtlicher Gesetze
* Gesetz zur Änderung der Bevorrechtigung der Verwendung elektrisch betriebener Fahrzeuge sowie damit zusammenhängender straßenverkehrsrechtlicher Vorschriften
* Gesetz zur Änderung des Tierschutzgesetzes und des 

In [18]:
json.dump({day: [gesetz['id'] for gesetz in value] for day, value in top.items()}, open('tagesordnung.json', 'wt'))

In [19]:
pre_process(stem('Erste Beratung des von der CDU/CSU-Fraktion eingebrachten Gesetzentwurfs zur rechtssicheren Einführung '))

'gesetz rechtssich einführung'

In [20]:
pre_process(stem(' – Zweite und dritte Beratung des von der Bundesregierung eingebrachten Entwurfs eines Gesetzes zur Durchführung'))

'gesetz durchführung'

In [21]:
pre_process(stem('a) Zweite und dritte Beratung des von der Bundesregierung eingebrachten Entwurfs eines Zweiten Gesetzes zur Änderung des Umweltstatistikgesetzes  '))

'zweit gesetz änderung umweltstatistikgesetz'